# Lesson 05: MIDIとシーケンサー - 既存曲の打ち込みと編曲

このレッスンでは、MIDIの基礎とシーケンサーを使った音楽制作を学びます。

**ゴール**: 既存の楽曲をMIDIデータとして打ち込み、メロディ・伴奏・ベース・ドラムを組み合わせたアレンジを作れるようになる。

## 目次
- 5.1 MIDI の基礎知識
- 5.2 シーケンサーで「きらきら星」を打ち込む
- 5.3 マルチトラック編曲 - きらきら星にベースとドラムを追加
- 5.4 リズムパターン - 伴奏の基本
- 5.5 コード進行 - 和声伴奏の基本
- 5.6 実践: 「ちょうちょ」のフルアレンジ
- 5.7 課題

In [ ]:
# 🛠️ 環境セットアップ
import sys
import os

# Google Colab環境かどうかを判定
try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

# ライブラリのセットアップ
if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    
    # 必要なパッケージをインストール
    !pip install numpy scipy matplotlib ipython japanize-matplotlib
    
    # GitHubからライブラリをクローン
    !git clone https://github.com/ggszk/simple-audio-programming.git
    
    # パスを追加
    sys.path.append('/content/simple-audio-programming')
    
    print("✅ セットアップ完了！")
    print("📝 このノートブックを自分用にコピーするには:")
    print("   ファイル → ドライブにコピーを保存")
    
else:
    print("📋 ローカル環境での前提条件:")
    print("   1. uvがインストールされていること")
    print("   2. プロジェクトディレクトリで 'uv sync --group dev' が実行済みであること")
    print("   3. 'uv run jupyter lab' で起動していること")
    
    print(f"\n🔍 デバッグ情報:")
    print(f"Python実行ファイル: {sys.executable}")
    print(f"作業ディレクトリ: {os.getcwd()}")
    
    try:
        import audio_lib
        print("✅ audio_lib ライブラリが利用可能です")
        print(f"audio_lib パス: {audio_lib.__file__}")
    except ImportError as e:
        print("❌ audio_lib ライブラリが見つかりません")
        print(f"エラー: {e}")
        print("📖 セットアップ手順:")
        print("   1. ターミナルでプロジェクトディレクトリに移動")
        print("   2. uv sync --group dev")
        print("   3. uv run jupyter lab")
        raise ImportError("uv環境が正しくセットアップされていません")

# インポート文
print("\n📦 必要なモジュールをimport中...")
try:
    from audio_lib import (
        sine_wave, square_wave, sawtooth_wave, adsr, save_audio, AudioSignal,
        note_to_frequency, note_name_to_number,
        Sequencer, Note, Track,  # シーケンサー関連クラス
        BasicPiano, BasicOrgan, BasicGuitar, BasicDrum  # 楽器クラス
    )
    print("✅ 全てのimportに成功しました")
    
    print(f"Note クラス: {Note}")
    print(f"Track クラス: {Track}")
    print(f"BasicPiano クラス: {BasicPiano}")
    
except ImportError as e:
    print(f"❌ import エラー: {e}")
    import traceback
    traceback.print_exc()

from IPython.display import Audio, display
import numpy as np
import matplotlib.pyplot as plt

# 日本語フォント設定（Colab用）
if IN_COLAB:
    import japanize_matplotlib
    print("✅ 日本語フォントを設定しました")

import warnings
warnings.filterwarnings('ignore')

print("\n🎹 MIDIとシーケンサーの学習を始めましょう！")

## 5.1 MIDI の基礎知識

**MIDI** (Musical Instrument Digital Interface) は、音楽の演奏データを表現する規格です。
音声そのものではなく、「何の音を」「どの強さで」「いつ」「どれだけの長さ」鳴らすかを記録します。

### MIDI ノート番号

| ノート番号 | 音名 | 説明 |
|-----------|------|------|
| 60 | C4 | 中央のド（ピアノの真ん中あたり） |
| 62 | D4 | レ |
| 64 | E4 | ミ |
| 65 | F4 | ファ |
| 67 | G4 | ソ |
| 69 | A4 | ラ (= 440Hz) |
| 71 | B4 | シ |
| 72 | C5 | 高いド |

### MIDI の主な要素

- **ノート番号**: 0-127 の値で音の高さを表す（数字が大きいほど高い音）
- **ベロシティ**: 0-127 の値で音の強さ（大きさ）を表す
- **開始時間**: 音が鳴り始めるタイミング（秒）
- **デュレーション**: 音の長さ（秒）

このライブラリでは、ノート番号の代わりに `"C4"` のような音名文字列も使えます。

In [ ]:
# MIDIノート番号と音名の対応を確認
from audio_lib import note_to_frequency, note_name_to_number, number_to_note_name

# C4 (中央のド) からC5 (高いド) まで
note_names = ["C4", "D4", "E4", "F4", "G4", "A4", "B4", "C5"]

print("音名  | MIDIノート番号 | 周波数 (Hz)")
print("-" * 40)
for name in note_names:
    num = note_name_to_number(name)
    freq = note_to_frequency(num)
    print(f" {name}  |      {num:3d}       | {freq:.1f}")

In [ ]:
# Note クラスの使い方
# 音名文字列でもMIDIノート番号でも指定できる

note1 = Note("C4", velocity=100, start_time=0.0, duration=0.5)
note2 = Note(60, velocity=100, start_time=0.0, duration=0.5)  # 同じ音

print(f"音名で作成:        {note1}")
print(f"ノート番号で作成:  {note2}")
print(f"周波数:            {note1.get_frequency():.1f} Hz")

## 5.2 シーケンサーで「きらきら星」を打ち込む

MIDIの基礎を理解したところで、実際に曲を打ち込んでみましょう。

最初の題材は誰でも知っている「きらきら星」（Twinkle Twinkle Little Star）です。

### きらきら星のメロディ (ハ長調)

```
ド ド ソ ソ ラ ラ ソ ー  ファ ファ ミ ミ レ レ ド ー
C  C  G  G  A  A  G  -   F    F   E  E  D  D  C  -
```

### シーケンサーの基本ワークフロー

1. `Sequencer` を作成
2. `Track` を作成し、音符 (`Note`) を追加
3. 楽器を設定
4. `render()` で音声に変換

In [ ]:
from audio_lib import Sequencer, Note, Track, create_simple_melody, BasicPiano

# 1. シーケンサーを作成
seq = Sequencer()

# 2. メロディトラックを作成
melody = Track(name="melody")

# 3. きらきら星のメロディを打ち込む（1フレーズ目 + 2フレーズ目）
#    各音符は4分音符 = 0.5秒、2分音符 = 1.0秒 とする
melody_notes = [
    # 1フレーズ目: ド ド ソ ソ ラ ラ ソ(伸ばし)
    ("C4", 100, 0.0, 0.5),
    ("C4", 100, 0.5, 0.5),
    ("G4", 100, 1.0, 0.5),
    ("G4", 100, 1.5, 0.5),
    ("A4", 100, 2.0, 0.5),
    ("A4", 100, 2.5, 0.5),
    ("G4", 100, 3.0, 1.0),   # 2分音符
    # 2フレーズ目: ファ ファ ミ ミ レ レ ド(伸ばし)
    ("F4", 100, 4.0, 0.5),
    ("F4", 100, 4.5, 0.5),
    ("E4", 100, 5.0, 0.5),
    ("E4", 100, 5.5, 0.5),
    ("D4", 100, 6.0, 0.5),
    ("D4", 100, 6.5, 0.5),
    ("C4", 100, 7.0, 1.0),   # 2分音符
]
melody.add_notes(melody_notes)

# 4. 楽器を設定してシーケンサーに追加
melody.set_instrument(BasicPiano())
seq.add_track(melody)

# 5. レンダリングして再生
audio = seq.render()
print(f"演奏時間: {len(audio.data) / 44100:.1f} 秒")
display(Audio(audio.data, rate=44100))

### create_simple_melody ヘルパー関数

同じ長さの音符が続く場合、`create_simple_melody` を使うとより簡潔に書けます。
`None` を入れると休符になります。

In [ ]:
# create_simple_melody を使った簡潔な書き方
seq2 = Sequencer()
melody2 = Track(name="melody")

# 音名のリストで打ち込み（全て同じ長さの4分音符）
# ソとドの後の休符(None)で2分音符のような間を表現
twinkle_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,  # ド ド ソ ソ ラ ラ ソ ー
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,  # ファ ファ ミ ミ レ レ ド ー
]

create_simple_melody(melody2, twinkle_notes, note_duration=0.5)
melody2.set_instrument(BasicPiano())
seq2.add_track(melody2)

audio2 = seq2.render()
display(Audio(audio2.data, rate=44100))

## 5.3 マルチトラック編曲 - きらきら星にベースとドラムを追加

メロディだけでは寂しいので、**ベース**と**ドラム**を追加してみましょう。

シーケンサーでは複数のトラックを重ねて同時に鳴らすことができます。

### ベースライン
メロディのコード（和音）のルート音を低い音域で鳴らします。
きらきら星のコード進行は C - G - F - C が基本です。

### ドラムパターン
ドラムは特別で、ノート番号が音の高さではなく打楽器の種類を表します:
- **36**: キックドラム（バスドラム）
- **38**: スネアドラム
- **42**: ハイハット

In [ ]:
from audio_lib import BasicGuitar, BasicDrum, create_chord

seq3 = Sequencer()

# --- メロディトラック（きらきら星 フル） ---
melody3 = Track(name="melody")
twinkle_full = [
    # 1-2小節: ド ド ソ ソ ラ ラ ソ ー
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    # 3-4小節: ファ ファ ミ ミ レ レ ド ー
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
    # 5-6小節: ソ ソ ファ ファ ミ ミ レ ー
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    # 7-8小節: ソ ソ ファ ファ ミ ミ レ ー
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    # 9-10小節: ド ド ソ ソ ラ ラ ソ ー (繰り返し)
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    # 11-12小節: ファ ファ ミ ミ レ レ ド ー
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody3, twinkle_full, note_duration=0.5)
melody3.set_instrument(BasicPiano())
melody3.volume = 0.7
seq3.add_track(melody3)

# --- ベーストラック ---
bass = Track(name="bass")

# コード進行に合わせたベースライン（1小節 = 2秒）
bass_pattern = [
    # セクションA (4小節)
    ("C2", 80, 0.0, 2.0),    # 1小節: C
    ("G2", 80, 2.0, 2.0),    # 2小節: G
    ("F2", 80, 4.0, 2.0),    # 3小節: F
    ("C2", 80, 6.0, 2.0),    # 4小節: C
    # セクションB (4小節)
    ("C2", 80, 8.0, 2.0),    # 5小節: C
    ("F2", 80, 10.0, 2.0),   # 6小節: F
    ("C2", 80, 12.0, 2.0),   # 7小節: C
    ("G2", 80, 14.0, 2.0),   # 8小節: G
    # セクションA再現 (4小節)
    ("C2", 80, 16.0, 2.0),
    ("G2", 80, 18.0, 2.0),
    ("F2", 80, 20.0, 2.0),
    ("C2", 80, 22.0, 2.0),
]
bass.add_notes(bass_pattern)
bass.set_instrument(BasicGuitar())
bass.volume = 0.5
seq3.add_track(bass)

# --- ドラムトラック ---
drums = Track(name="drums")

# 基本的な4ビートパターンを全体に繰り返す
total_bars = 12  # 12小節
beat_duration = 0.5  # 4分音符 = 0.5秒

for bar in range(total_bars):
    bar_start = bar * 4 * beat_duration  # 1小節 = 4拍
    for beat in range(4):
        t = bar_start + beat * beat_duration
        # ハイハット: 毎拍
        drums.add_note(42, 70, t, 0.25)
        # キック: 1拍目と3拍目
        if beat == 0 or beat == 2:
            drums.add_note(36, 90, t, 0.3)
        # スネア: 2拍目と4拍目
        if beat == 1 or beat == 3:
            drums.add_note(38, 85, t, 0.25)

drums.set_instrument(BasicDrum())
drums.volume = 0.4
seq3.add_track(drums)

# レンダリングして再生
audio3 = seq3.render()
print(f"きらきら星 (メロディ + ベース + ドラム)")
print(f"演奏時間: {len(audio3.data) / 44100:.1f} 秒")
print(f"トラック数: {len(seq3.tracks)}")
display(Audio(audio3.data, rate=44100))

## 5.4 リズムパターン - 伴奏の基本

ドラムのリズムパターンは曲の雰囲気を大きく左右します。
ここではよく使われるパターンを関数化して、曲の伴奏として使えるようにしましょう。

### 代表的なリズムパターン

- **8ビート**: ロック・ポップスの基本。ハイハットが8分音符で刻む
- **16ビート**: ファンク・R&B。より細かいハイハット
- **ワルツ (3拍子)**: クラシック・民謡。1拍目にキック、2-3拍目にハイハット

In [ ]:
def create_drum_pattern(track, pattern_type="8beat", bars=4, beat_duration=0.5):
    """
    ドラムパターンを生成してトラックに追加する関数
    
    Args:
        track: ドラムトラック
        pattern_type: パターンの種類 ("8beat", "16beat", "waltz")
        bars: 小節数
        beat_duration: 1拍の長さ(秒)
    """
    if pattern_type == "8beat":
        beats_per_bar = 4
        for bar in range(bars):
            bar_start = bar * beats_per_bar * beat_duration
            for beat in range(beats_per_bar):
                t = bar_start + beat * beat_duration
                # ハイハット: 8分音符刻み（各拍を2分割）
                track.add_note(42, 70, t, 0.15)
                track.add_note(42, 60, t + beat_duration / 2, 0.15)
                # キック: 1拍目と3拍目
                if beat == 0 or beat == 2:
                    track.add_note(36, 90, t, 0.3)
                # スネア: 2拍目と4拍目
                if beat == 1 or beat == 3:
                    track.add_note(38, 85, t, 0.2)
    
    elif pattern_type == "16beat":
        beats_per_bar = 4
        for bar in range(bars):
            bar_start = bar * beats_per_bar * beat_duration
            for beat in range(beats_per_bar):
                t = bar_start + beat * beat_duration
                # ハイハット: 16分音符刻み（各拍を4分割）
                for sub in range(4):
                    vel = 70 if sub == 0 else 50
                    track.add_note(42, vel, t + sub * beat_duration / 4, 0.1)
                # キック: 1拍目と3拍目
                if beat == 0 or beat == 2:
                    track.add_note(36, 90, t, 0.3)
                # スネア: 2拍目と4拍目
                if beat == 1 or beat == 3:
                    track.add_note(38, 85, t, 0.2)
    
    elif pattern_type == "waltz":
        beats_per_bar = 3
        for bar in range(bars):
            bar_start = bar * beats_per_bar * beat_duration
            # 1拍目: キック
            track.add_note(36, 90, bar_start, 0.3)
            # 2拍目, 3拍目: ハイハット
            track.add_note(42, 70, bar_start + beat_duration, 0.15)
            track.add_note(42, 70, bar_start + 2 * beat_duration, 0.15)

print("ドラムパターン関数を定義しました")
print("使い方: create_drum_pattern(track, pattern_type='8beat', bars=4)")

In [ ]:
# 各パターンを聴き比べ
for ptype in ["8beat", "16beat", "waltz"]:
    seq_demo = Sequencer()
    drum_track = Track(name="drums")
    create_drum_pattern(drum_track, pattern_type=ptype, bars=2)
    drum_track.set_instrument(BasicDrum())
    seq_demo.add_track(drum_track)
    audio_demo = seq_demo.render()
    print(f"\n--- {ptype} パターン ---")
    display(Audio(audio_demo.data, rate=44100))

## 5.5 コード進行 - 和声伴奏の基本

コード（和音）はメロディに厚みを与え、曲の雰囲気を決定します。

### よく使われるコード進行

ポップスの王道コード進行を紹介します（ハ長調の場合）:

| 進行名 | コード | 構成音 |
|-------|--------|--------|
| I (トニック) | C | ド - ミ - ソ |
| IV (サブドミナント) | F | ファ - ラ - ド |
| V (ドミナント) | G | ソ - シ - レ |
| vi (マイナー) | Am | ラ - ド - ミ |

**I - V - vi - IV** (C - G - Am - F) は数え切れないほどのヒット曲で使われている進行です。

### `create_chord` ヘルパー関数

`create_chord(track, chord_notes, start_time, duration)` で和音をトラックに追加できます。

In [ ]:
# コード進行を伴奏として作る関数
def create_chord_accompaniment(track, chord_progression, bars_per_chord=1, beat_duration=0.5):
    """
    コード進行を伴奏としてトラックに追加
    
    Args:
        track: コードトラック
        chord_progression: コードのリスト。各コードは音名のリスト
            例: [["C3","E3","G3"], ["G3","B3","D4"], ...]
        bars_per_chord: 1コードあたりの小節数
        beat_duration: 1拍の長さ(秒)
    """
    bar_duration = 4 * beat_duration  # 4拍で1小節
    chord_duration = bars_per_chord * bar_duration
    
    for i, chord_notes in enumerate(chord_progression):
        start = i * chord_duration
        create_chord(track, chord_notes, start_time=start, duration=chord_duration, velocity=70)

# I - V - vi - IV 進行をオルガンで鳴らしてみる
seq_chord = Sequencer()

chord_track = Track(name="chords")
progression = [
    ["C3", "E3", "G3"],   # C (I)
    ["G3", "B3", "D4"],   # G (V)
    ["A3", "C4", "E4"],   # Am (vi)
    ["F3", "A3", "C4"],   # F (IV)
]
create_chord_accompaniment(chord_track, progression)
chord_track.set_instrument(BasicOrgan())
chord_track.volume = 0.5
seq_chord.add_track(chord_track)

audio_chord = seq_chord.render()
print("コード進行: C - G - Am - F (I - V - vi - IV)")
display(Audio(audio_chord.data, rate=44100))

In [ ]:
# きらきら星にコード伴奏を追加した完全版
seq_full = Sequencer()

# メロディ
melody_full = Track(name="melody")
twinkle_complete = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    "G4", "G4", "F4", "F4", "E4", "E4", "D4", None,
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_full, twinkle_complete, note_duration=0.5)
melody_full.set_instrument(BasicPiano())
melody_full.volume = 0.7
seq_full.add_track(melody_full)

# コード伴奏 (1コード = 1小節 = 2秒)
chords_full = Track(name="chords")
twinkle_chords = [
    # セクションA (4小節)
    ["C3", "E3", "G3"],   # C
    ["G3", "B3", "D4"],   # G
    ["F3", "A3", "C4"],   # F
    ["C3", "E3", "G3"],   # C
    # セクションB (4小節)
    ["C3", "E3", "G3"],   # C
    ["F3", "A3", "C4"],   # F
    ["C3", "E3", "G3"],   # C
    ["G3", "B3", "D4"],   # G
    # セクションA再現 (4小節)
    ["C3", "E3", "G3"],   # C
    ["G3", "B3", "D4"],   # G
    ["F3", "A3", "C4"],   # F
    ["C3", "E3", "G3"],   # C
]
create_chord_accompaniment(chords_full, twinkle_chords)
chords_full.set_instrument(BasicOrgan())
chords_full.volume = 0.3
seq_full.add_track(chords_full)

# ベース
bass_full = Track(name="bass")
bass_notes_full = [
    # セクションA
    ("C2", 80, 0.0, 2.0), ("G2", 80, 2.0, 2.0),
    ("F2", 80, 4.0, 2.0), ("C2", 80, 6.0, 2.0),
    # セクションB
    ("C2", 80, 8.0, 2.0), ("F2", 80, 10.0, 2.0),
    ("C2", 80, 12.0, 2.0), ("G2", 80, 14.0, 2.0),
    # セクションA再現
    ("C2", 80, 16.0, 2.0), ("G2", 80, 18.0, 2.0),
    ("F2", 80, 20.0, 2.0), ("C2", 80, 22.0, 2.0),
]
bass_full.add_notes(bass_notes_full)
bass_full.set_instrument(BasicGuitar())
bass_full.volume = 0.4
seq_full.add_track(bass_full)

# ドラム
drums_full = Track(name="drums")
create_drum_pattern(drums_full, pattern_type="8beat", bars=12)
drums_full.set_instrument(BasicDrum())
drums_full.volume = 0.35
seq_full.add_track(drums_full)

# レンダリング
audio_full = seq_full.render()
print("きらきら星 - フルアレンジ版")
print(f"  メロディ: ピアノ")
print(f"  コード:   オルガン")
print(f"  ベース:   ギター")
print(f"  ドラム:   8ビート")
print(f"  演奏時間: {len(audio_full.data) / 44100:.1f} 秒")
display(Audio(audio_full.data, rate=44100))

## 5.6 実践: 「ちょうちょ」のフルアレンジ

ここまで学んだテクニックを組み合わせて、別の曲を最初からアレンジしてみましょう。

### 「ちょうちょ」のメロディ (ハ長調)

```
ソ ミ ミ ー  ファ レ レ ー  ド レ ミ ファ  ソ ソ ソ ー
G  E  E  -   F   D  D  -   C  D  E  F    G  G  G  -

ソ ミ ミ ー  ファ レ レ ー  ド ミ ソ ソ  ミ ミ ミ ー
G  E  E  -   F   D  D  -   C  E  G  G   E  E  E  -
```

### 手順
1. メロディを打ち込む
2. コード進行を決める
3. ベースラインを追加
4. ドラムパターンを追加
5. 音量バランスを調整

In [ ]:
# ステップ1: メロディの打ち込み
seq_chou = Sequencer()

melody_chou = Track(name="melody")

# 「ちょうちょ」のメロディ
choucho_notes = [
    # 1-2小節: ソ ミ ミ ー ファ レ レ ー
    "G4", "E4", "E4", None, "F4", "D4", "D4", None,
    # 3-4小節: ド レ ミ ファ ソ ソ ソ ー
    "C4", "D4", "E4", "F4", "G4", "G4", "G4", None,
    # 5-6小節: ソ ミ ミ ー ファ レ レ ー
    "G4", "E4", "E4", None, "F4", "D4", "D4", None,
    # 7-8小節: ド ミ ソ ソ ミ ミ ミ ー
    "C4", "E4", "G4", "G4", "E4", "E4", "E4", None,
]

create_simple_melody(melody_chou, choucho_notes, note_duration=0.5)
melody_chou.set_instrument(BasicPiano())
melody_chou.volume = 0.7
seq_chou.add_track(melody_chou)

# まずメロディだけで確認
audio_chou_melody = seq_chou.render()
print("ちょうちょ - メロディのみ")
display(Audio(audio_chou_melody.data, rate=44100))

In [ ]:
# ステップ2-3: コード伴奏 + ベースを追加

# コード進行
chords_chou = Track(name="chords")
choucho_chords = [
    ["C3", "E3", "G3"],   # 1小節: C
    ["G2", "B2", "D3"],   # 2小節: G
    ["C3", "E3", "G3"],   # 3小節: C
    ["C3", "E3", "G3"],   # 4小節: C
    ["C3", "E3", "G3"],   # 5小節: C
    ["G2", "B2", "D3"],   # 6小節: G
    ["C3", "E3", "G3"],   # 7小節: C
    ["C3", "E3", "G3"],   # 8小節: C
]
create_chord_accompaniment(chords_chou, choucho_chords)
chords_chou.set_instrument(BasicOrgan())
chords_chou.volume = 0.3
seq_chou.add_track(chords_chou)

# ベース: コードのルート音
bass_chou = Track(name="bass")
bass_chou_notes = [
    ("C2", 80, 0.0, 2.0),   # 1小節
    ("G2", 80, 2.0, 2.0),   # 2小節
    ("C2", 80, 4.0, 2.0),   # 3小節
    ("C2", 80, 6.0, 2.0),   # 4小節
    ("C2", 80, 8.0, 2.0),   # 5小節
    ("G2", 80, 10.0, 2.0),  # 6小節
    ("C2", 80, 12.0, 2.0),  # 7小節
    ("C2", 80, 14.0, 2.0),  # 8小節
]
bass_chou.add_notes(bass_chou_notes)
bass_chou.set_instrument(BasicGuitar())
bass_chou.volume = 0.4
seq_chou.add_track(bass_chou)

print("ちょうちょ - メロディ + コード + ベース")
audio_chou_2 = seq_chou.render()
display(Audio(audio_chou_2.data, rate=44100))

In [ ]:
# ステップ4-5: ドラムを追加して完成

drums_chou = Track(name="drums")
create_drum_pattern(drums_chou, pattern_type="8beat", bars=8)
drums_chou.set_instrument(BasicDrum())
drums_chou.volume = 0.35
seq_chou.add_track(drums_chou)

# 完成版をレンダリング
audio_chou_final = seq_chou.render()

print("=" * 50)
print("  ちょうちょ - フルアレンジ版")
print("=" * 50)
print(f"  メロディ: ピアノ (音量 0.7)")
print(f"  コード:   オルガン (音量 0.3)")
print(f"  ベース:   ギター (音量 0.4)")
print(f"  ドラム:   8ビート (音量 0.35)")
print(f"  演奏時間: {len(audio_chou_final.data) / 44100:.1f} 秒")
print("=" * 50)
display(Audio(audio_chou_final.data, rate=44100))

## 5.7 課題

ここまでの内容を使って、以下の課題に取り組んでみましょう。

### 課題1: 好きな既存曲の冒頭8小節をメロディとして打ち込む

知っている曲（童謡、アニメ主題歌、J-POPなど）の冒頭部分を選び、
メロディをノートデータとして打ち込んでください。

**ヒント:**
- まず曲を口ずさんで、音の上下関係を把握する
- ハ長調（#や♭なし）に移調すると打ち込みやすい
- 最初は4分音符だけで近似して、後でリズムを調整する

### 課題2: その曲に伴奏（コード + ベース + ドラム）を付ける

課題1で作ったメロディに対して、以下を追加してください:
- コード伴奏（オルガンかピアノで）
- ベースライン（コードのルート音を使う）
- ドラムパターン（曲の雰囲気に合ったパターンを選ぶ）

**ヒント:**
- わからない場合は「曲名 コード進行」で検索すると見つかることが多い
- まずは I (C), IV (F), V (G) の3つのコードだけで試してみる
- 各トラックの `volume` を調整してバランスを取る

In [ ]:
# 課題1: メロディの打ち込み
# ここに好きな曲のメロディを打ち込んでみましょう

seq_kadai = Sequencer()

melody_kadai = Track(name="melody")

# 例: 「かえるの合唱」の冒頭
# ド レ ミ ファ ミ レ ド ー  ミ ファ ソ ラ ソ ファ ミ ー
kadai_notes = [
    "C4", "D4", "E4", "F4", "E4", "D4", "C4", None,
    "E4", "F4", "G4", "A4", "G4", "F4", "E4", None,
    # ここに続きを追加...
]

create_simple_melody(melody_kadai, kadai_notes, note_duration=0.5)
melody_kadai.set_instrument(BasicPiano())
melody_kadai.volume = 0.7
seq_kadai.add_track(melody_kadai)

audio_kadai = seq_kadai.render()
print("課題1: メロディ")
display(Audio(audio_kadai.data, rate=44100))

In [ ]:
# 課題2: 伴奏を追加
# メロディに合うコード・ベース・ドラムを追加しましょう

# コード伴奏
chords_kadai = Track(name="chords")
kadai_chords = [
    ["C3", "E3", "G3"],   # C
    ["G2", "B2", "D3"],   # G
    # ここにコード進行を追加...
]
create_chord_accompaniment(chords_kadai, kadai_chords)
chords_kadai.set_instrument(BasicOrgan())
chords_kadai.volume = 0.3
seq_kadai.add_track(chords_kadai)

# ベース
bass_kadai = Track(name="bass")
bass_kadai_notes = [
    ("C2", 80, 0.0, 2.0),
    ("G2", 80, 2.0, 2.0),
    # ここにベースラインを追加...
]
bass_kadai.add_notes(bass_kadai_notes)
bass_kadai.set_instrument(BasicGuitar())
bass_kadai.volume = 0.4
seq_kadai.add_track(bass_kadai)

# ドラム
drums_kadai = Track(name="drums")
create_drum_pattern(drums_kadai, pattern_type="8beat", bars=4)
drums_kadai.set_instrument(BasicDrum())
drums_kadai.volume = 0.35
seq_kadai.add_track(drums_kadai)

# 完成版
audio_kadai_full = seq_kadai.render()
print("課題2: フルアレンジ")
display(Audio(audio_kadai_full.data, rate=44100))

## 宿題

自分の好きな曲を1曲選び、以下を提出してください:

1. **曲名と選んだ理由**
2. **メロディの打ち込み** (最低8小節)
3. **コード進行の付与**
4. **ベース + ドラムを含むフルアレンジ**

打ち込む際のコツ:
- 楽譜が手に入る場合は参考にする
- 楽譜がない場合は、実際に曲を聴きながら音を探る（耳コピ）
- 完璧でなくてOK。まずは雰囲気が伝わるレベルを目指す
- 曲のキー（調）がわからない場合はハ長調に移調して打ち込む

---

### このレッスンのまとめ

- MIDIは「何の音を、いつ、どれだけの長さで鳴らすか」を記録する規格
- `Note` でMIDI音符を、`Track` でパートを、`Sequencer` で楽曲全体を管理
- メロディ、コード、ベース、ドラムを組み合わせてアレンジを作る
- 既存曲の打ち込みは音楽理論を実践的に学ぶ最良の方法